# Dieses Notebook dient der Konsolidierung einzelner DataFrames aus den Datenquellen

Weiterhin wird jedes der 3 finalen DataFrames (df_solldaten_cis, df_istdaten_emr, df_istdaten_prc) so normalisiert und vorbereitet, dass alle Mafi Touren entfernt werden, Kennzeichen sowie Fahrer normalisiert werden, die Rohdatenquellen auf die relevanten Inhalte reduziert werden, die Zeitstempel entsprechend geparst werden und die Daten letztlich auf Tour-oder Stopp-Ebene vorliegen.

## 1. Struktur der PRC-Dateien sichten
Die PRC-Dateien sind heterogenes XML: Eine einzelne Datei kann mehrere Meldungen unterschiedlichen Typs enthalten. Bevor wir einlesen, verschaffen wir uns einen Überblick, welche Meldungstypen überhaupt vorkommen und wie sie aufgebaut sind - je ein vollständiges Beispiel pro Typ.

In [16]:
import re
import pandas as pd
from pathlib import Path

# Anzeigeoptionen, damit DataFrames vollständig sichtbar sind
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", None)

BASE_PATH = Path.cwd().parent / "data" / "raw"

# Alle PRC-Dateien finden
prc_files = sorted(BASE_PATH.glob("msg_*.prc"))
print(f"Gefundene PRC-Dateien: {len(prc_files)}")
print()

# Bekannte Meldungstypen (laut Doku/Beobachtung)
bekannte_typen = ["Sendposstatus", "Stationsstatus", "FMSStatus", "Tourstatus", "Position"]

# Pro Typ die erste Datei finden, in der er vorkommt
beispiele = {}
for pfad in prc_files:
    with open(pfad, "r", encoding="utf-8") as f:
        inhalt = f.read()
    for typ in bekannte_typen:
        if typ not in beispiele and f"<{typ}>" in inhalt:
            beispiele[typ] = (pfad.name, inhalt)
    if len(beispiele) == len(bekannte_typen):
        break

print(f"Beispiele gefunden für {len(beispiele)} von {len(bekannte_typen)} Typen.")
print()

for typ, (name, inhalt) in beispiele.items():
    print("=" * 80)
    print(f"Meldungstyp: {typ}    (Datei: {name})")
    print("=" * 80)
    print(inhalt)
    print()

Gefundene PRC-Dateien: 24521

Beispiele gefunden für 5 von 5 Typen.

Meldungstyp: Position    (Datei: msg_20260228000043_956.imp.20260228000310889.prc)
﻿<Import>
  <Position>
    <GeoPosition>
      <PositionsID>16049334470</PositionsID>
      <Longitude>9.98548</Longitude>
      <Latitude>53.52335</Latitude>
      <Zeitstempel_GPS>2026-02-27T23:59:59+01:00</Zeitstempel_GPS>
      <Richtung>0</Richtung>
      <Geschwindigkeit>0</Geschwindigkeit>
      <KMStand>339687</KMStand>
    </GeoPosition>
    <Zeitstempel_Fahrzeug>2026-02-28T00:00:00+01:00</Zeitstempel_Fahrzeug>
    <Zeitstempel_Server>2026-02-28T00:00:43</Zeitstempel_Server>
    <FahrzeugID>Plö-TS856</FahrzeugID>
  </Position>
</Import>

Meldungstyp: FMSStatus    (Datei: msg_20260301141707_983.imp.20260301141842821.prc)
﻿<Import>
  <Tourstatus>
    <GeoPosition>
      <PositionsID>16053897222</PositionsID>
      <Longitude>9.98927</Longitude>
      <Latitude>53.52262</Latitude>
      <Zeitstempel_GPS>2026-03-01T14:13:25+01:00</

## 2. Einlesen und Zusammenführen der PRC-Meldungen
Wir lesen alle Dateien ein und erzeugen eine Zeile je Meldungselement unter `<Import>`. Übernommen werden nur die Felder, die für den Vergleich mit Soll/EMR und für die Routen-/Standzeitanalyse gebraucht werden: Meldungstyp, Fahrzeug, die drei Zeitstempel, GPS (Position, Geschwindigkeit, Kilometerstand), der Status sowie die Schlüssel `TourID`, `StationID` und `SendposID` — Letztere vollständig mit Suffixen und `TP`-Bestandteil, da genau diese die Re-Dispositionen und Stationspositionen abbilden. Die `Werte`-Inhalte (FMS-Bussignale, Sendungsmengen) werden nicht übernommen: Die Signale haben keine Vergleichsquelle, und die Mengen/Gewichte sind in den Soll-Daten je Tour konstant. Der Ereignistyp (`Ereignis_Typ`) wird gezielt nur als direktes Kind des Meldungselements gezogen, damit er nicht mit dem gleichnamigen `Typ` innerhalb der `Werte` kollidiert.

In [17]:
import xml.etree.ElementTree as ET

# GeoPosition-Felder für Geofencing/Route/Standzeit
GEO_FELDER = ["PositionsID", "Longitude", "Latitude", "Geschwindigkeit", "KMStand", "Zeitstempel_GPS"]
# Direkte Felder unter dem Meldungselement
MELDUNG_FELDER = ["Zeitstempel_Fahrzeug", "Zeitstempel_Server", "FahrzeugID",
                  "Status", "TourID", "StationID", "SendposID"]

def parse_prc_datei(pfad):
    """Eine Zeile je Meldungselement unter <Import>. Erfasst nur die für Vergleich und
    Routen-/Standzeitanalyse benötigten Felder; die Werte-Inhalte (FMS-Signale,
    Sendungsmengen) werden bewusst nicht übernommen."""
    zeilen = []
    root = ET.parse(pfad).getroot()          # <Import>; ET verarbeitet das BOM selbst
    for meldung in root:                      # jedes Kind = eine Meldung
        eintrag = {"msg_typ": meldung.tag, "quelle_datei": pfad.name}
        geo = meldung.find("GeoPosition")
        for feld in GEO_FELDER:
            eintrag[feld] = geo.findtext(feld) if geo is not None else None
        for feld in MELDUNG_FELDER:
            eintrag[feld] = meldung.findtext(feld)
        # Ereignistyp nur als DIREKTES Kind des Meldungselements (z. B. <Ereignis><Typ>),
        # damit das <Typ> innerhalb von <Werte> (FMSStatus/Sendposstatus) nicht mitgezogen wird.
        eintrag["Ereignis_Typ"] = meldung.findtext("Typ")
        zeilen.append(eintrag)
    return zeilen


alle_zeilen = []
fehlerhafte_dateien = []

for pfad in prc_files:
    try:
        alle_zeilen.extend(parse_prc_datei(pfad))
    except ET.ParseError as e:
        fehlerhafte_dateien.append((pfad.name, str(e)))

prc_raw = pd.DataFrame(alle_zeilen)

print(f"Verarbeitete Dateien:     {len(prc_files)}")
print(f"Nicht parsebare Dateien:  {len(fehlerhafte_dateien)}")
for name, fehler in fehlerhafte_dateien[:5]:
    print(f"   - {name}: {fehler}")
print(f"Erzeugte Meldungszeilen:  {len(prc_raw)}")
print()
print("Meldungstypen:")
print(prc_raw["msg_typ"].value_counts(dropna=False))

prc_raw.head()

Verarbeitete Dateien:     24521
Nicht parsebare Dateien:  0
Erzeugte Meldungszeilen:  33155

Meldungstypen:
msg_typ
Position          11535
FMSStatus          8634
Stationsstatus     8299
Sendposstatus      3280
Tourstatus         1405
Ereignis              2
Name: count, dtype: int64


,msg_typ,quelle_datei,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,FahrzeugID,Status,TourID,StationID,SendposID,Ereignis_Typ
0,Position,msg_20260228000043_956.imp.20260228000310889.prc,16049334470,9.98548,53.52335,0,339687,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:00:43,Plö-TS856,NaN,NaN,NaN,NaN,NaN
1,Position,msg_20260228000043_957.imp.20260228000310923.prc,16049339872,9.98613,53.5234,0,502579,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:00:43,TS859,NaN,NaN,NaN,NaN,NaN
2,Position,msg_20260228000315_958.imp.20260228000815933.prc,16049349544,9.98619,53.52356,0,183989,2026-02-27T20:53:12+01:00,2026-02-28T00:02:42+01:00,2026-02-28T00:03:15,OD-TS 8050,NaN,NaN,NaN,NaN,NaN
3,Position,msg_20260228000345_959.imp.20260228000815958.prc,16049351179,9.98592,53.52345,0,461200,2026-02-27T23:59:59+01:00,2026-02-28T00:00:00+01:00,2026-02-28T00:03:45,OD-TS 8046,NaN,NaN,NaN,NaN,NaN
4,Position,msg_20260228060024_960.imp.20260228060348749.prc,16050228689,9.98616,53.52342,0,502579,2026-02-28T05:59:59+01:00,2026-02-28T06:00:00+01:00,2026-02-28T06:00:24,TS859,NaN,NaN,NaN,NaN,NaN


### 2.1 Erste Untersuchung: der Meldungstyp `Ereignis`
Direkt nach dem Parsen zeigt der Output zweierlei: alle 24.521 Dateien wurden ohne Fehler verarbeitet, und neben den fünf erwarteten Typen taucht ein sechster auf - `Ereignis` mit nur zwei Vorkommen im gesamten Monat. Das sehen wir uns zuerst an. Unser flaches Schema hat dafür nur die gemeinsamen Felder erfasst; ein `Ereignis` trägt laut Mapping-XML aber einen eigenen Ereignistyp (z. B. Pause, Stau, Panne, Tanken). Deshalb betrachten wir zusätzlich das rohe XML der beiden Quelldateien, um den vollständigen Inhalt zu kennen, bevor wir entscheiden, wie wir damit umgehen.

In [18]:
# Erste Untersuchung: der unerwartete sechste Meldungstyp "Ereignis" (2 Zeilen)
ereignis_zeilen = prc_raw[prc_raw["msg_typ"] == "Ereignis"]
display(ereignis_zeilen)

# Unser flaches Schema erfasst nur die gemeinsamen Felder. Um zu sehen, was eine
# Ereignis-Meldung tatsächlich enthält, sehen wir uns das rohe XML der Quelldateien an.
for name in ereignis_zeilen["quelle_datei"].unique():
    print("=" * 80)
    print(name)
    print("=" * 80)
    print((BASE_PATH / name).read_text(encoding="utf-8-sig"))

,msg_typ,quelle_datei,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,FahrzeugID,Status,TourID,StationID,SendposID,Ereignis_Typ
4629,Ereignis,msg_20260305081857_4412.imp.20260305082219283.prc,16083444083,9.96948,53.51279,0,461908,2026-03-05T08:15:03+01:00,2026-03-05T08:15:03+01:00,2026-03-05T08:18:57,OD-TS 8046,NaN,NaN,NaN,NaN,3
21442,Ereignis,msg_20260319132430_16778.imp.20260319132701458.prc,16183750851,9.92304,53.59549,0,807024,2026-03-19T13:24:28+01:00,2026-03-19T13:24:28+01:00,2026-03-19T13:24:30,WL-PL431,NaN,NaN,NaN,NaN,3


msg_20260305081857_4412.imp.20260305082219283.prc
<Import>
  <FMSStatus>
    <GeoPosition>
      <PositionsID>16083444083</PositionsID>
      <Longitude>9.96948</Longitude>
      <Latitude>53.51279</Latitude>
      <Zeitstempel_GPS>2026-03-05T08:15:03+01:00</Zeitstempel_GPS>
      <Richtung>0</Richtung>
      <Geschwindigkeit>0</Geschwindigkeit>
      <KMStand>461908</KMStand>
    </GeoPosition>
    <Zeitstempel_Fahrzeug>2026-03-05T08:15:03+01:00</Zeitstempel_Fahrzeug>
    <Zeitstempel_Server>2026-03-05T08:18:57</Zeitstempel_Server>
    <FahrzeugID>OD-TS 8046</FahrzeugID>
    <FMSStatusID>16083444083</FMSStatusID>
    <Werte>
      <Typ>4</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>1</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>5</Typ>
      <Wert>0</Wert>
    </Werte>
    <Werte>
      <Typ>2</Typ>
      <Wert>461908.531</Wert>
    </Werte>
    <Werte>
      <Typ>4</Typ>
      <Wert>0</Wert>
    </Werte>
  </FMSStatus>
  <Ereignis>
    <GeoPosition

Beide `Ereignis`-Meldungen tragen statt eines `Status` ein eigenes Feld `Typ` (hier jeweils `3`) und eine `EreignisID`. Laut Mapping-XML entspricht `Typ 3` dem Ereignis „Beginn-Stau". Es handelt sich also um fahrzeugbezogene Ereignismeldungen — hier zweimal ein Stau-Beginn — ohne Status- und ohne Tour-/Stationsbindung, jeweils zusammen mit einer FMSStatus-Meldung in derselben Datei. Das erklärt auch, warum bei diesen Zeilen `Status`, `TourID`, `StationID` und `SendposID` leer sind: Diese Felder existieren in einer Ereignismeldung nicht.

Für die Standzeit- und Routenabweichungsanalyse sind sie nicht verwertbar — sie lassen sich keiner Station oder Tour zuordnen und kommen mit zwei Vorkommen im ganzen Monat viel zu selten vor. Wir entfernen die beiden Zeilen. Die ereignisspezifischen Felder (`Typ`, `EreignisID`) hat unser flaches Schema bewusst nicht erfasst; da der Typ ohnehin entfällt, ist das ohne Belang.

### 2.2 Technische Erstprüfung der PRC-Meldungen
Wie bei den anderen Quellen verschaffen wir uns zunächst einen technischen Überblick. `.info()` zeigt die Spalten, ihre Datentypen und die Belegung (Non-Null) und ist der Ausgangspunkt für die anschließende Bereinigung.

In [19]:
prc_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 33155 entries, 0 to 33154
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   msg_typ               33155 non-null  str  
 1   quelle_datei          33155 non-null  str  
 2   PositionsID           33155 non-null  str  
 3   Longitude             33155 non-null  str  
 4   Latitude              33155 non-null  str  
 5   Geschwindigkeit       33155 non-null  str  
 6   KMStand               33155 non-null  str  
 7   Zeitstempel_GPS       33155 non-null  str  
 8   Zeitstempel_Fahrzeug  33155 non-null  str  
 9   Zeitstempel_Server    33155 non-null  str  
 10  FahrzeugID            33155 non-null  str  
 11  Status                12984 non-null  str  
 12  TourID                1405 non-null   str  
 13  StationID             8299 non-null   str  
 14  SendposID             3280 non-null   str  
 15  Ereignis_Typ          2 non-null      str  
dtypes: str(16)
memo

## 3. Bereinigung und Vorbereitung
Aus der Erstprüfung folgt der erste Schritt: Alle Spalten liegen als Text vor und werden in die passenden Typen überführt. Wir legen dafür mit `prc` eine Arbeitskopie an, `prc_raw` bleibt unverändert.

Die drei Zeitstempel sind der heikle Teil. `Zeitstempel_GPS` und `Zeitstempel_Fahrzeug` tragen einen Zeitzonen-Offset (`+01:00` in der Winter-, `+02:00` ab der Sommerzeit), `Zeitstempel_Server` ist offset-los. Wir parsen die beiden ersten über UTC und rechnen sie auf lokale Zeit (Europe/Berlin) um, anschließend ohne Zeitzone — so liegen alle drei als lokale, tz-lose Zeit vor und sind mit den ebenfalls tz-losen Zeiten aus Soll und EMR vergleichbar. Die Sommerzeit-Umstellung Ende März wird dabei automatisch korrekt behandelt.

Bei den numerischen Feldern werden Längen-/Breitengrad, Geschwindigkeit und Kilometerstand zu Zahlen. `Status` und `Ereignis_Typ` werden als nullable Integer (`Int64`) geführt, da sie nur bei ihren jeweiligen Meldungstypen belegt sind. Die ID-Felder (`PositionsID`, `TourID`, `StationID`, `SendposID`) bleiben bewusst Text.

Alle Umwandlungen laufen mit `errors="coerce"`, sodass nicht-parsebare Werte als Leerwert sichtbar würden. 

In [20]:
prc = prc_raw.copy()

# Zeitstempel: GPS/Fahrzeug tragen einen Offset (+01:00/+02:00 je nach Sommerzeit),
# Server ist offset-los. Wir bringen alle drei auf lokale, tz-lose Zeit (Europe/Berlin),
# damit sie mit den tz-losen Zeiten aus Soll und EMR vergleichbar sind.
for col in ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug"]:
    prc[col] = (pd.to_datetime(prc[col], utc=True, errors="coerce")
                  .dt.tz_convert("Europe/Berlin").dt.tz_localize(None))
prc["Zeitstempel_Server"] = pd.to_datetime(prc["Zeitstempel_Server"], errors="coerce")

# Numerische Felder
for col in ["Longitude", "Latitude", "Geschwindigkeit", "KMStand"]:
    prc[col] = pd.to_numeric(prc[col], errors="coerce")
# Statuscode und Ereignistyp als nullable Integer (bleiben leer, wo der Typ sie nicht trägt)
for col in ["Status", "Ereignis_Typ"]:
    prc[col] = pd.to_numeric(prc[col], errors="coerce").astype("Int64")

print("Datentypen nach Umwandlung:")
print(prc.dtypes)
print()
print("Fehlgeschlagene Umwandlungen (immer belegte Spalten):")
for col in ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug", "Zeitstempel_Server",
            "Longitude", "Latitude", "Geschwindigkeit", "KMStand"]:
    print(f"  {col}: {prc[col].isna().sum()}")
print()
print("Belegung nach Umwandlung (Kontrolle, soll unverändert sein):")
print(f"  Status:       {prc['Status'].notna().sum()}   (erwartet 12984)")
print(f"  Ereignis_Typ: {prc['Ereignis_Typ'].notna().sum()}   (erwartet 2)")

Datentypen nach Umwandlung:
msg_typ                            str
quelle_datei                       str
PositionsID                        str
Longitude                      float64
Latitude                       float64
Geschwindigkeit                  int64
KMStand                          int64
Zeitstempel_GPS         datetime64[us]
Zeitstempel_Fahrzeug    datetime64[us]
Zeitstempel_Server      datetime64[us]
FahrzeugID                         str
Status                           Int64
TourID                             str
StationID                          str
SendposID                          str
Ereignis_Typ                     Int64
dtype: object

Fehlgeschlagene Umwandlungen (immer belegte Spalten):
  Zeitstempel_GPS: 0
  Zeitstempel_Fahrzeug: 0
  Zeitstempel_Server: 0
  Longitude: 0
  Latitude: 0
  Geschwindigkeit: 0
  KMStand: 0

Belegung nach Umwandlung (Kontrolle, soll unverändert sein):
  Status:       12984   (erwartet 12984)
  Ereignis_Typ: 2   (erwartet 2)


Mit den geparsten Zeitstempeln prüfen wir den Zeitbereich der PRC-Daten. Daraus ergibt sich, ob und wie wir auf den gemeinsamen Analysezeitraum (März) filtern müssen. Wir sehen uns alle drei Zeitstempel an, um zu entscheiden, welcher als fachlicher Meldungszeitpunkt für den Filter dient.

In [21]:
print("Zeitbereich je Zeitstempel-Spalte:")
for col in ["Zeitstempel_GPS", "Zeitstempel_Fahrzeug", "Zeitstempel_Server"]:
    print(f"  {col}: {prc[col].min()}  bis  {prc[col].max()}")

print("\nMeldungen je Kalendertag (Zeitstempel_Fahrzeug):")
print(prc["Zeitstempel_Fahrzeug"].dt.date.value_counts().sort_index())

Zeitbereich je Zeitstempel-Spalte:
  Zeitstempel_GPS: 2026-02-09 12:19:59  bis  2026-03-31 13:25:19
  Zeitstempel_Fahrzeug: 2026-02-28 00:00:00  bis  2026-03-31 13:25:22
  Zeitstempel_Server: 2026-02-28 00:00:43  bis  2026-03-31 13:26:09

Meldungen je Kalendertag (Zeitstempel_Fahrzeug):
Zeitstempel_Fahrzeug
2026-02-28      18
2026-03-01      24
2026-03-02    1452
2026-03-03    1232
2026-03-04    1340
2026-03-05    1498
2026-03-06    1150
2026-03-07      24
2026-03-08      24
2026-03-09    1446
2026-03-10    1772
2026-03-11    1930
2026-03-12    1722
2026-03-13    1388
2026-03-14      33
2026-03-15      21
2026-03-16    1590
2026-03-17    1778
2026-03-18    1476
2026-03-19    1851
2026-03-20    1184
2026-03-21      53
2026-03-22      27
2026-03-23    1343
2026-03-24    1243
2026-03-25    1548
2026-03-26    1846
2026-03-27     860
2026-03-28      38
2026-03-29      30
2026-03-30    1482
2026-03-31    1732
Name: count, dtype: int64


### 3.2 Bezugszeitpunkt und Filter auf den Analysezeitraum
Der Zeitbereich zeigt zweierlei. Erstens reichen die Zeitstempel unterschiedlich weit zurück: `Zeitstempel_GPS` beginnt bereits am 09.02., während `Zeitstempel_Fahrzeug` und `Zeitstempel_Server` erst am 28.02. einsetzen. Das bestätigt, dass einzelne GPS-Zeitstempel veraltet sein können (gepufferte Positionen) und sich nicht als Meldungszeitpunkt eignen. Wir verwenden daher `Zeitstempel_Fahrzeug` als fachlichen Meldungszeitpunkt — den Zeitpunkt, zu dem das Fahrzeug die Meldung erzeugt hat, am nächsten an der `Meldungszeit` aus den EMR-Daten.

Zweitens beginnen die Daten Ende Februar. Wie bei Soll und EMR begrenzen wir auf den gemeinsamen Analysezeitraum März und entfernen die 18 Meldungen vom 28.02.

In der Tagesverteilung fällt zudem auf, dass einzelne Tage nur rund 20–50 Meldungen tragen gegenüber etwa 1.200–1.900 an Werktagen. Könnte an Mafi liegen, wird noch rausgefiltert.

In [22]:
# Zeitstempel_Fahrzeug als fachlicher Meldungszeitpunkt (Erzeugung im Fahrzeug),
# analog zur Meldungszeit bei EMR. Filter auf den gemeinsamen Analysezeitraum März.
vorher = len(prc)
prc = prc[(prc["Zeitstempel_Fahrzeug"] >= "2026-03-01") &
          (prc["Zeitstempel_Fahrzeug"] <  "2026-04-01")].copy()

print(f"Zeilen vorher:      {vorher}")
print(f"Vor März entfernt:  {vorher - len(prc)}")
print(f"Zeilen im März:     {len(prc)}")

Zeilen vorher:      33155
Vor März entfernt:  18
Zeilen im März:     33137


### 3.3 Fahrzeugkennungen
Wir sehen uns zunächst die rohen Fahrzeugkennungen an. Daraus ergibt sich, welche Schreibvarianten vorkommen, ob dieselbe Normalisierung wie bei Soll und EMR greift und ob ein Mafi-Fahrzeug enthalten ist, das wie in den anderen Quellen zu entfernen wäre. Anders als Soll und EMR tragen die PRC-Meldungen keinen Fahrernamen; der Fahrzeug-Schlüssel ist hier die einzige fahrzeugbezogene Verknüpfung.

In [23]:
print("Eindeutige Fahrzeugkennungen:", prc["FahrzeugID"].nunique())
prc["FahrzeugID"].value_counts(dropna=False)

Eindeutige Fahrzeugkennungen: 13


FahrzeugID
PI-EN 444     4390
TS859         3794
Plö-TS853     3113
OD-TS 8041    3044
OD-TS 8048    2965
OD-TS 8050    2855
Plö-TS856     2647
WL-PL431      2579
OD-TS 8046    2351
Mafi          1665
OD-TS 8044    1638
PI-EN 900     1267
Plo-TS857      829
Name: count, dtype: int64

Die Prüfung zeigt 13 Kennungen, darunter wieder `Mafi` mit 1.665 Meldungen. Wie in Soll und EMR entfernen wir das Mafi-Fahrzeug gemäß Vorgabe von Frank. Die übrigen Kennungen bringen wir mit derselben Funktion wie in den anderen Quellen auf ein einheitliches Format (Großschreibung, Umlaute aufgelöst, Leerzeichen entfernt, Buchstaben- und Zifferngruppen mit Bindestrich getrennt) und wenden den bestätigten Alias `TS859 → PLO-TS-859` an. Abschließend benennen wir die Spalte in `LKW_KENNZ` um, damit sie zum gemeinsamen Schlüssel von Soll und EMR passt. Es verbleiben 12 Fahrzeuge mit 31.472 Meldungen.

In [24]:
import re

def normalisiere_kennzeichen(wert):
    """Vereinheitlicht Fahrzeugkennungen: Großschreibung, Umlaute aufgelöst,
    Buchstaben-/Zifferngruppen mit Bindestrich getrennt (wie in Soll und EMR)."""
    if pd.isna(wert):
        return pd.NA
    wert = str(wert).upper().strip()
    wert = wert.replace("Ä", "A").replace("Ö", "O").replace("Ü", "U").replace("ß", "SS")
    return "-".join(re.findall(r"[A-Z]+|[0-9]+", wert))


# Mafi-Fahrzeug entfernen (wie in Soll und EMR, Vorgabe von Frank)
vorher = len(prc)
prc = prc[prc["FahrzeugID"] != "Mafi"].copy()
print(f"Mafi-Zeilen entfernt: {vorher - len(prc)}")

# Kennzeichen normalisieren + bestätigten Alias anwenden
prc["FahrzeugID"] = prc["FahrzeugID"].apply(normalisiere_kennzeichen).replace({"TS-859": "PLO-TS-859"})

# Auf gemeinsamen Spaltennamen bringen
prc = prc.rename(columns={"FahrzeugID": "LKW_KENNZ"})

print(f"Zeilen jetzt: {len(prc)}")
print("Eindeutige LKW_KENNZ:", prc["LKW_KENNZ"].nunique())
print(prc["LKW_KENNZ"].value_counts().sort_index())

Mafi-Zeilen entfernt: 1665
Zeilen jetzt: 31472
Eindeutige LKW_KENNZ: 12
LKW_KENNZ
OD-TS-8041    3044
OD-TS-8044    1638
OD-TS-8046    2351
OD-TS-8048    2965
OD-TS-8050    2855
PI-EN-444     4390
PI-EN-900     1267
PLO-TS-853    3113
PLO-TS-856    2647
PLO-TS-857     829
PLO-TS-859    3794
WL-PL-431     2579
Name: count, dtype: int64


## 4. Vertiefte Datenprüfung
Vor dem Export sehen wir uns die notierten Auffälligkeiten genauer an wie mögliche Dubletten, den Kilometerstand und die Plausibilität der Geodaten.

### 4.1 Dubletten
Wie bei den anderen Quellen prüfen wir auf doppelte Zeilen. Zu beachten: `PositionsID` ist nicht pro Zeile eindeutig, weil sich die gebündelten Meldungen einer Datei (z. B. Tourstatus und FMSStatus) denselben GPS-Fix teilen. Eine echte Dublette wäre daher eine inhaltlich vollständig identische Zeile. Es gibt keine exakten Voll-Dubletten, aber 2.042 inhaltsgleiche Zeilen, die sich nur in der Quelldatei unterscheiden — dieselbe Meldung wurde also in mehr als einer Datei importiert. Bevor wir bereinigen, sehen wir uns an, welche Meldungstypen betroffen sind und ob es sich tatsächlich um identische Wiederholungen handelt.

In [28]:
inhalt_cols = [c for c in prc.columns if c != "quelle_datei"]

dubletten = (prc[prc.duplicated(subset=inhalt_cols, keep=False)]
             .sort_values(["PositionsID", "msg_typ", "quelle_datei"]))

print("Betroffene Zeilen insgesamt:", len(dubletten))
print()
print("Verteilung der inhaltsgleichen Dubletten nach Meldungstyp:")
print(dubletten["msg_typ"].value_counts(dropna=False))
print()
display(dubletten[["msg_typ", "quelle_datei", "PositionsID", "Zeitstempel_Fahrzeug",
                   "Status", "StationID", "SendposID"]].head(12))

Betroffene Zeilen insgesamt: 3155

Verteilung der inhaltsgleichen Dubletten nach Meldungstyp:
msg_typ
FMSStatus         2985
Stationsstatus     170
Name: count, dtype: int64



,msg_typ,quelle_datei,PositionsID,Zeitstempel_Fahrzeug,Status,StationID,SendposID
143,FMSStatus,msg_20260302054940_1063.imp.20260302055135386.prc,16056058618,2026-03-02 05:49:36,<NA>,NaN,NaN
145,FMSStatus,msg_20260302054940_1064.imp.20260302055135419.prc,16056058618,2026-03-02 05:49:36,<NA>,NaN,NaN
157,FMSStatus,msg_20260302055042_1070.imp.20260302055135882.prc,16056063396,2026-03-02 05:50:17,<NA>,NaN,NaN
159,FMSStatus,msg_20260302055042_1071.imp.20260302055135966.prc,16056063396,2026-03-02 05:50:17,<NA>,NaN,NaN
161,FMSStatus,msg_20260302055042_1072.imp.20260302055135999.prc,16056063396,2026-03-02 05:50:17,<NA>,NaN,NaN
182,FMSStatus,msg_20260302060251_1087.imp.20260302060645331.prc,16056162705,2026-03-02 05:59:12,<NA>,NaN,NaN
184,FMSStatus,msg_20260302060251_1088.imp.20260302060645445.prc,16056162705,2026-03-02 05:59:12,<NA>,NaN,NaN
186,FMSStatus,msg_20260302060251_1089.imp.20260302060645467.prc,16056162705,2026-03-02 05:59:12,<NA>,NaN,NaN
205,FMSStatus,msg_20260302060626_1101.imp.20260302060646144.prc,16056184183,2026-03-02 06:05:48,<NA>,NaN,NaN
207,FMSStatus,msg_20260302060626_1102.imp.20260302060646214.prc,16056184183,2026-03-02 06:05:48,<NA>,NaN,NaN


Die inhaltsgleichen Zeilen betreffen zwei Meldungstypen: 2.985 FMSStatus und 170 Stationsstatus. An den Quelldateinamen ist erkennbar, dass dieselbe Meldung in mehreren aufeinanderfolgenden Dateien desselben Import-Vorgangs ankam (fortlaufende Nummern, Importzeit im Millisekundenabstand) - es sind echte Wiederholungen derselben Meldung, keine eigenständigen Ereignisse. Manche Meldungen wurden zwei- bis viermal importiert.

Besonders relevant sind die 170 Stationsstatus-Dubletten: Ein doppelt erfasstes Stationsereignis (z. B. eine Ankunft) würde später eine Standzeit doppelt zählen. Wir entfernen daher die inhaltsgleichen Wiederholungen und behalten je Meldung eine Zeile.

In [30]:
vorher = len(prc)
prc = prc.drop_duplicates(subset=inhalt_cols).copy()

print(f"Zeilen vorher:           {vorher}")
print(f"Inhaltsgleiche entfernt: {vorher - len(prc)}")
print(f"Zeilen jetzt:            {len(prc)}")
print()
print("Verbleibende inhaltsgleiche Dubletten:",
      prc.duplicated(subset=inhalt_cols).sum())

Zeilen vorher:           29430
Inhaltsgleiche entfernt: 0
Zeilen jetzt:            29430

Verbleibende inhaltsgleiche Dubletten: 0


### 4.2 Übersicht: fehlende Werte, eindeutige Werte und Statistik
In der Erstprüfung hatten wir nur `.info()` ausgeführt. Hier ergänzen wir die Übersicht analog zu Soll und EMR: Datentypen, fehlende Werte und eindeutige Werte je Spalte sowie die Statistik der numerischen Felder. Die fehlenden Werte bestätigen erneut die typgebundene Belegung (Status, IDs und Ereignistyp nur bei ihren Meldungstypen). Aus der Statistik lesen wir anschließend den Kilometerstand und die Geodaten.

In [31]:
prc_overview = pd.DataFrame({
    "datentyp": prc.dtypes.astype(str),
    "fehlende_werte": prc.isna().sum(),
    "fehlende_werte_%": (prc.isna().mean() * 100).round(2),
    "eindeutige_werte": prc.nunique(dropna=True),
})
display(prc_overview)

print("Statistik der numerischen Felder:")
display(prc[["Longitude", "Latitude", "Geschwindigkeit", "KMStand"]].describe())

,datentyp,fehlende_werte,fehlende_werte_%,eindeutige_werte
msg_typ,str,0,0.00,6
quelle_datei,str,0,0.00,22753
PositionsID,str,0,0.00,19831
Longitude,float64,0,0.00,8462
Latitude,float64,0,0.00,7635
Geschwindigkeit,int64,0,0.00,118
KMStand,int64,0,0.00,5868
Zeitstempel_GPS,datetime64[us],0,0.00,16459
Zeitstempel_Fahrzeug,datetime64[us],0,0.00,16108
Zeitstempel_Server,datetime64[us],0,0.00,9876


Statistik der numerischen Felder:


,Longitude,Latitude,Geschwindigkeit,KMStand
count,29430.000000,29430.000000,29430.000000,29430.000000
mean,9.959396,53.567364,13.517193,284361.587734
std,0.344904,0.190482,22.629036,223996.851720
min,8.165830,51.298860,0.000000,0.000000
25%,9.956840,53.522630,0.000000,179093.000000
50%,9.985700,53.523530,0.000000,195984.000000
75%,9.992027,53.627730,23.000000,462479.000000
max,13.647510,54.781390,128.000000,808364.000000


### 4.3 Kilometerstand
Die Statistik zeigt `min = 0` bei einem Median um 196.000 — es gibt also 0-Werte im Kilometerstand. Wir sehen uns an, bei welchen Meldungstypen diese auftreten, denn daraus ergibt sich, ob es ein Datenfehler ist oder Bauart einzelner Meldungstypen.

In [32]:
n_null = (prc["KMStand"] == 0).sum()
print(f"KMStand == 0: {n_null}  ({n_null / len(prc) * 100:.1f}%)")
print()
print("0-Werte nach Meldungstyp:")
print(prc[prc["KMStand"] == 0]["msg_typ"].value_counts())
print()
print("KMStand je Meldungstyp (min / median / max):")
print(prc.groupby("msg_typ")["KMStand"].agg(["min", "median", "max"]))

KMStand == 0: 5533  (18.8%)

0-Werte nach Meldungstyp:
msg_typ
Position          2565
Stationsstatus    1755
Sendposstatus     1002
Tourstatus         211
Name: count, dtype: int64

KMStand je Meldungstyp (min / median / max):
                   min    median     max
msg_typ                                 
Ereignis        461908  634466.0  807024
FMSStatus       143450  315344.0  808361
Position             0  196324.0  808364
Sendposstatus        0  194994.0  808360
Stationsstatus       0  186750.0  808361
Tourstatus           0  194994.0  807996


Der erste Blick widerlegt die naheliegende Vermutung, dass die 0-Werte an einem einzelnen Meldungstyp hängen: Sie treten bei Position, Stationsstatus, Sendposstatus und Tourstatus auf, während dieselben Typen sonst plausible Kilometerstände tragen (Median rund 187.000–196.000). Nur FMSStatus und Ereignis haben durchgehend echte Werte. Die 0er sind also ein quer durch die Typen laufender Teilbestand. Um sie einzuordnen, sehen wir uns an, ob sie sich auf bestimmte Fahrzeuge konzentrieren und ob sie mit Stillstand zusammenfallen.

In [33]:
print("KMStand=0 Anteil je Fahrzeug (%):")
print(prc.groupby("LKW_KENNZ")["KMStand"]
        .apply(lambda s: (s == 0).mean() * 100).round(1).sort_values(ascending=False))

nullkm = prc[prc["KMStand"] == 0]
print(f"\nVon {len(nullkm)} Meldungen mit KMStand=0 haben {(nullkm['Geschwindigkeit'] == 0).sum()} "
      f"die Geschwindigkeit 0.")

KMStand=0 Anteil je Fahrzeug (%):
LKW_KENNZ
OD-TS-8044    100.0
PLO-TS-853    100.0
PLO-TS-857    100.0
OD-TS-8041      0.0
OD-TS-8048      0.0
OD-TS-8046      0.0
PI-EN-444       0.0
OD-TS-8050      0.0
PI-EN-900       0.0
PLO-TS-856      0.0
PLO-TS-859      0.0
WL-PL-431       0.0
Name: KMStand, dtype: float64

Von 5533 Meldungen mit KMStand=0 haben 2835 die Geschwindigkeit 0.


Die 0-Werte sind fahrzeugbedingt: Drei Fahrzeuge (OD-TS-8044, PLO-TS-853, PLO-TS-857) melden den Kilometerstand durchgehend als 0, die übrigen neun durchgehend mit echten Werten — ein Entweder-oder ohne Übergänge. Es ist also keine situative Stillstandslücke, sondern ein Telematik-Merkmal dieser drei Fahrzeuge, deren Kilometerstand nicht übertragen wird (dass 2.835 der 0-Meldungen bei Geschwindigkeit 0 liegen, ist nur ein Nebeneffekt).

Wir entfernen nichts: GPS-Spur, Zeiten und Status dieser Fahrzeuge sind vollständig, nur der Kilometerstand fehlt. Für die spätere Kilometer-Abweichung heißt das, dass für diese drei Fahrzeuge aus PRC kein Kilometerstand vorliegt. Zusammen mit dem aus EMR bekannten Punkt (Kilometerstand dort nicht monoton steigend) ist der Kilometerstand insgesamt nur eingeschränkt für die Distanzauswertung nutzbar — das klären wir in der Analysephase.

### 4.4 Geodaten
Die Statistik zeigt Längengrad 8,17–13,65 und Breitengrad 51,30–54,78, alles innerhalb Deutschlands und ohne 0-Koordinaten. Damit sind grobe Fehler (Nullinsel, vertauschte oder fehlende Werte) bereits ausgeschlossen. Zur Bestätigung sehen wir uns die Extrempunkte an und prüfen, ob die weit vom Großraum Hamburg entfernten Meldungen echte Fahrten sind — also viele Meldungen über mehrere Fahrzeuge — oder vereinzelte Ausreißer.

In [34]:
print("Extrempunkte (min/max Längen- und Breitengrad):")
extrem = pd.concat([
    prc.nsmallest(1, "Longitude"), prc.nlargest(1, "Longitude"),
    prc.nsmallest(1, "Latitude"),  prc.nlargest(1, "Latitude"),
])
display(extrem[["msg_typ", "LKW_KENNZ", "Zeitstempel_Fahrzeug",
                "Longitude", "Latitude", "Geschwindigkeit"]])

# Meldungen außerhalb des Großraums Hamburg: echte Fahrten oder Einzelausreißer?
fern = prc[(prc["Longitude"] > 11) | (prc["Longitude"] < 9) |
           (prc["Latitude"] < 52.5) | (prc["Latitude"] > 54)]
print(f"Meldungen außerhalb des Großraums Hamburg: {len(fern)}  ({len(fern) / len(prc) * 100:.1f}%)")
print("davon je Fahrzeug:")
print(fern["LKW_KENNZ"].value_counts())

Extrempunkte (min/max Längen- und Breitengrad):


,msg_typ,LKW_KENNZ,Zeitstempel_Fahrzeug,Longitude,Latitude,Geschwindigkeit
2192,Position,OD-TS-8050,2026-03-03 10:26:18,8.16583,53.28381,11
16650,Stationsstatus,OD-TS-8050,2026-03-16 19:27:11,13.64751,54.14740,0
32928,Position,WL-PL-431,2026-03-31 11:50:06,11.99148,51.29886,26
23632,Stationsstatus,PLO-TS-856,2026-03-23 09:06:43,8.83709,54.78139,0


Meldungen außerhalb des Großraums Hamburg: 1162  (3.9%)
davon je Fahrzeug:
LKW_KENNZ
OD-TS-8048    279
PLO-TS-856    264
OD-TS-8050    257
PLO-TS-853    173
WL-PL-431      55
OD-TS-8041     50
OD-TS-8044     50
PLO-TS-857     34
Name: count, dtype: int64


Die Extrempunkte sind reale Fahrzeuge zu plausiblen Zeiten an stimmigen Orten: Raum Oldenburg (fahrend), die Ostsee-Ecke Richtung Lubmin (an einer Station), der Raum Halle/Leipzig (fahrend) und der äußerste Norden Richtung Nordfriesland (an einer Station). Die 1.162 Meldungen außerhalb des Großraums Hamburg (3,9 %) verteilen sich über acht Fahrzeuge mit jeweils dutzenden bis hunderten Meldungen — also echte Langstreckentouren, keine Einzelausreißer. Die Geodaten sind damit plausibel; wir nehmen keine Änderung vor.

## 5. Finalisierung und Export
Die Hilfsspalte `quelle_datei`, die wir nur zur Nachverfolgung mitgeführt haben, entfernen wir vor dem Export. Den fertigen Frame benennen wir als `df_istdaten_prc_clean` und schreiben ihn nach `data/processed/`, gleiche Konvention wie bei Soll und EMR. Zusätzlich zur CSV speichern wir eine Pickle-Datei: Sie erhält die Datentypen (`Int64`-Statuscodes, `datetime`-Zeitstempel), die beim erneuten Einlesen einer CSV verloren gingen — das erleichtert das Zusammenführen im vierten Notebook.

In [35]:
# quelle_datei war nur zur Nachverfolgung -> vor dem Export entfernen
df_istdaten_prc_clean = prc.drop(columns=["quelle_datei"]).reset_index(drop=True)

# Export, gleiche Konvention wie Soll/EMR
ausgabe = BASE_PATH.parent / "processed"
ausgabe.mkdir(parents=True, exist_ok=True)
df_istdaten_prc_clean.to_csv(ausgabe / "istdaten_prc_clean.csv", index=False, encoding="utf-8-sig")
df_istdaten_prc_clean.to_pickle(ausgabe / "istdaten_prc_clean.pkl")

# Abschlusskontrolle
print("Shape:", df_istdaten_prc_clean.shape)
print("\nSpalten und Datentypen:")
print(df_istdaten_prc_clean.dtypes)
print("\nExportiert nach:", ausgabe)
df_istdaten_prc_clean.head()

Shape: (29430, 15)

Spalten und Datentypen:
msg_typ                            str
PositionsID                        str
Longitude                      float64
Latitude                       float64
Geschwindigkeit                  int64
KMStand                          int64
Zeitstempel_GPS         datetime64[us]
Zeitstempel_Fahrzeug    datetime64[us]
Zeitstempel_Server      datetime64[us]
LKW_KENNZ                          str
Status                           Int64
TourID                             str
StationID                          str
SendposID                          str
Ereignis_Typ                     Int64
dtype: object

Exportiert nach: c:\Users\jonas\OneDrive\Dokumente\FH_Kiel\VS_Code\Python_Projekte\hts_cargo_project\data\processed


,msg_typ,PositionsID,Longitude,Latitude,Geschwindigkeit,KMStand,Zeitstempel_GPS,Zeitstempel_Fahrzeug,Zeitstempel_Server,LKW_KENNZ,Status,TourID,StationID,SendposID,Ereignis_Typ
0,Position,16052606718,9.98615,53.52342,0,502579,2026-02-28 23:59:59,2026-03-01 00:00:00,2026-03-01 00:00:37,PLO-TS-859,<NA>,NaN,NaN,NaN,<NA>
1,Position,16052611797,9.98547,53.52335,0,339687,2026-02-28 23:59:59,2026-03-01 00:00:00,2026-03-01 00:00:37,PLO-TS-856,<NA>,NaN,NaN,NaN,<NA>
2,Position,16052619744,9.98589,53.52336,0,183989,2026-02-28 23:58:35,2026-03-01 00:02:42,2026-03-01 00:03:09,OD-TS-8050,<NA>,NaN,NaN,NaN,<NA>
3,Position,16052620670,9.98853,53.52292,0,461200,2026-02-28 23:59:59,2026-03-01 00:00:00,2026-03-01 00:03:40,OD-TS-8046,<NA>,NaN,NaN,NaN,<NA>
4,Position,16053108200,9.98546,53.52335,0,339687,2026-03-01 05:59:59,2026-03-01 06:00:00,2026-03-01 06:00:36,PLO-TS-856,<NA>,NaN,NaN,NaN,<NA>
